In [116]:
import os
import random
import warnings
from pathlib import Path
from io import BytesIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import requests
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from torchvision import models

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [117]:
def seed_everything(seed):
    random.seed(seed) # фиксируем генератор случайных чисел
    os.environ['PYTHONHASHSEED'] = str(seed) # фиксируем заполнения хешей
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU
    #torch.backends.cudnn.deterministic = True # выбираем только детерминированные алгоритмы (для сверток)
    #torch.backends.cudnn.benchmark = False # фиксируем алгоритм вычисления сверток


seed_everything(42)
#Сид из 15 семинара

Класс конфигурации 

In [118]:
class CFG:
    seed = 42
    image_col = "image_link"
    target_col = "price"
    cache_dir = "data/images_cache"
    img_size = 224
    batch_size = 32
    num_workers = 0 #количество активных процессов на загрузку данных
    epochs = 5
    lr = 1e-4
    val_size = 0.2 #Процент валидационной выборки
    artifacts_dir = "model_outputs/resnet18" #где сохраняем артифакты
    model_name = "resnet18_best.pt"

In [119]:
train_df = pd.read_csv("data/raw/train.csv")
test_df = pd.read_csv("data/raw/test.csv")
train_df.head()

,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49


Разделим наш train_test на 2 части train и validation

In [120]:
val_data = train_df.iloc[:int(len(train_df) * 0.2)].reset_index(drop=True)
train_data = train_df.iloc[int(len(train_df) * 0.2):].reset_index(drop=True)

В нашей задаче нужно достать кучу картинок из ссылок и сохранить их.
Тут используем requests, мы проходили это на нашем курсе, но именно таких задач не решили. 

In [121]:
from urllib.request import urlretrieve
from pathlib import Path
from tqdm.auto import tqdm

def download_images_simple(df):
    Path(CFG.cache_dir).mkdir(parents=True, exist_ok=True)

    image_paths = []

    for url in tqdm(df["image_link"], desc="Downloading images"):
        filename = url.split("/")[-1]
        image_path = f"{CFG.cache_dir}/{filename}"

        try:
            if not Path(image_path).exists():
                urlretrieve(url, image_path)

            image_paths.append(image_path)

        except:
            image_paths.append(None)

    df = df.copy()
    df["image_path"] = image_paths
    df = df.dropna(subset=["image_path"]).reset_index(drop=True)

    return df

In [122]:
class ImagePriceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.cache_dir = Path(CFG.cache_dir)

    def __len__(self):
        return len(self.df)

    def get_image_filename(self, url):
        return str(url).split("/")[-1].split("?")[0]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        url = row["image_link"]
        filename = self.get_image_filename(url)
        image_path = self.cache_dir / filename

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        price = np.log1p(row["price"]).astype("float32")

        return image, torch.tensor(price, dtype=torch.float32)

Выполним трансформацию, код взят из семинара 18-19 DL Techniques. 

In [123]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize(256), # меняем размер
    T.RandomCrop(CFG.img_size), # вырезаем случайную область указанного размера
    T.RandomHorizontalFlip(), # зеркальное отражение по горизонтали
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD) # нормализация
])

valid_transform = T.Compose([ # на валидации нельзя делать никаких преобразований (кроме изменения размера),
                               # ведь мы должны проверить модель на реальных картинках
    T.Resize(256),
    T.CenterCrop(CFG.img_size),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

Собираем датасеты, то есть открываем картинку, принимаем трансформацию и возвращаем
Потом загружаем данные, собираем картинки в батч, перемешиваем.

In [124]:
train_dataset = ImagePriceDataset(train_data, transform=train_transform)
val_dataset = ImagePriceDataset(val_data, transform=valid_transform)
test_dataset = ImagePriceDataset(test_df, transform=valid_transform)
train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers)

val_loader = DataLoader(val_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers)
test_loader = DataLoader(test_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers)

Посмотрим, скок картинок в батче, каналы, размер

In [125]:
images, targets = next(iter(train_loader))
print(images.shape)

torch.Size([32, 3, 224, 224])


Наконец сама модель ResNet18 она обычно используется для задачи классификации, но т.к нам нужно предсказывать цену, то последний слой модели был заменен на регрессию.

https://www.kaggle.com/code/sreesankar711/skew-ml-resnet-18

In [126]:
class ResNet18PriceRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.resnet18(weights=None)
        in_features = self.model.fc.in_features
        # Заменяем последний слой на один выход для цены
        self.model.fc = nn.Linear(in_features, 1)

    def forward(self, x):
        return self.model(x).squeeze(1)


model_resnet18 = ResNet18PriceRegressor().to(device)
model_resnet18

ResNet18PriceRegressor(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, e

Зададим, как модель будет учиться. Введем функцию ошибки, оптимизатор (обновление весов) и размер шага.

In [127]:
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model_resnet18.parameters(),
    lr=CFG.lr,
    weight_decay=1e-5
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

In [128]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0

    for images, targets in tqdm(loader, desc="Train", leave=False):
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(loader.dataset)

    return epoch_loss

In [ ]:
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0

    preds = []
    targets_all = []

    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Valid", leave=False):
            images = images.to(device)
            targets = targets.to(device)

            outputs = model(images)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * images.size(0)

            preds.extend(outputs.detach().cpu().numpy())
            targets_all.extend(targets.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    preds = np.array(preds)
    targets_all = np.array(targets_all)

    preds_price = np.expm1(preds)
    targets_price = np.expm1(targets_all)

    preds_price = np.clip(preds_price, 0, None)

    mae = mean_absolute_error(targets_price, preds_price)

    mse = mean_squared_error(targets_price, preds_price)
    rmse = np.sqrt(mse)

    r2 = r2_score(targets_price, preds_price)

    metrics = {
        "val_loss": epoch_loss,
        "mae": mae,
        "rmse": rmse,
        "r2": r2
    }

    return metrics

In [130]:
def fit_model(model, train_loader, val_loader, optimizer, scheduler, criterion, device):
    best_rmse = float("inf")
    history = []

    best_model_path = os.path.join(CFG.artifacts_dir, CFG.model_name)

    for epoch in range(1, CFG.epochs + 1):
        print(f"\nEpoch {epoch}/{CFG.epochs}")

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device
        )

        val_metrics = validate_one_epoch(
            model,
            val_loader,
            criterion,
            device
        )

        scheduler.step(val_metrics["val_loss"])

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["val_loss"],
            "mae": val_metrics["mae"],
            "rmse": val_metrics["rmse"],
            "r2": val_metrics["r2"]
        }

        history.append(row)

        print(
            f"train_loss: {train_loss:.4f} | "
            f"val_loss: {val_metrics['val_loss']:.4f} | "
            f"MAE: {val_metrics['mae']:.2f} | "
            f"RMSE: {val_metrics['rmse']:.2f} | "
            f"R2: {val_metrics['r2']:.4f}"
        )

        if val_metrics["rmse"] < best_rmse:
            best_rmse = val_metrics["rmse"]

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "epoch": epoch,
                    "best_rmse": best_rmse,
                },
                best_model_path
            )

            print(f"Best model saved: {best_model_path}")

    history = pd.DataFrame(history)

    history_path = os.path.join(CFG.artifacts_dir, "resnet18_history.csv")
    history.to_csv(history_path, index=False)

    return model, history

In [131]:
model_resnet18, history_resnet18 = fit_model(
    model=model_resnet18,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device
)


Epoch 1/5


Train:   0%|          | 0/1875 [00:00<?, ?it/s]

KeyboardInterrupt: 